In [1]:
import yfinance as yf
import pandas as pd
import os

In [2]:
# Daily USD/PHP from 2018 to today
vix = yf.download("^VIX", start="2015-01-01", end="2026-05-04", 
                 interval="1d", auto_adjust=False, multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [3]:
vix

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2015-01-02,17.790001,17.790001,20.139999,17.049999,17.760000,0
2015-01-05,19.920000,19.920000,21.290001,19.190001,19.190001,0
2015-01-06,21.120001,21.120001,22.900000,19.520000,20.330000,0
2015-01-07,19.309999,19.309999,20.719999,19.040001,20.150000,0
2015-01-08,17.010000,17.010000,18.090000,16.990000,17.930000,0
...,...,...,...,...,...,...
2026-04-27,18.020000,18.020000,19.270000,18.020000,19.209999,0
2026-04-28,17.830000,17.830000,19.430000,17.780001,18.299999,0
2026-04-29,18.809999,18.809999,19.000000,17.809999,17.830000,0


In [4]:
vix.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2849 entries, 2015-01-02 to 2026-05-01
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  2849 non-null   float64
 1   Close      2849 non-null   float64
 2   High       2849 non-null   float64
 3   Low        2849 non-null   float64
 4   Open       2849 non-null   float64
 5   Volume     2849 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 155.8 KB


In [5]:
vix.isnull().sum()

Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

In [6]:
df_vix = vix[["Close"]].rename(columns={"Close": "vix"})

In [7]:
df_vix

,vix
Date,
2015-01-02,17.790001
2015-01-05,19.920000
2015-01-06,21.120001
2015-01-07,19.309999
2015-01-08,17.010000
...,...
2026-04-27,18.020000
2026-04-28,17.830000
2026-04-29,18.809999


In [8]:
df_vix = df_vix.reset_index() # date is alr in datetime
df_vix.columns = ["date", "vix"]
df_vix ["date"] = pd.to_datetime(df_vix ["date"])

In [9]:
df_vix = df_vix.set_index("date").sort_index()

In [10]:
df_vix = df_vix.resample("W-TUE").agg(
    vix_close=("vix", "last"),
    vix_mean=("vix", "mean")
)
# weekly.index = weekly.index + pd.Timedelta(days=3)
# weekly.index.name = "week_tuesday"

In [11]:
df_vix

,vix_close,vix_mean
date,,
2015-01-06,21.120001,19.610001
2015-01-13,20.559999,18.806000
2015-01-20,19.889999,21.177500
2015-01-27,17.219999,16.930000
2015-02-03,17.330000,19.386000
...,...,...
2026-04-07,25.780001,24.590001
2026-04-14,18.360001,19.448000
2026-04-21,19.500000,18.392000


In [12]:
df_vix.to_parquet("../data/semi-cleaned/weekly_vix.parquet")